In [4]:
import pandas as pd

df = pd.read_csv('../../temp/tie_records.csv')
df

,full_address_std,tie_matched_addresses,tie_latitudes,tie_longitudes,tie_geoids
0,"1485 BAYSHORE BLVD, SAN FRANCISCO, CA 94124, U...","1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 ...",37.725535736384 | 37.725535736384,-122.401401989518 | -122.401401989518,060750233002003 | 060750233002003
1,"3130 LA SELVA DR, SAN MATEO, CA 94403, UNITED ...","3130 LA SELVA CIR, SAN MATEO, CA, 94403 | 3130...",37.542678745951 | 37.54307237241,-122.284437924233 | -122.285084558557,060816084003008 | 060816084003001
2,"26 PIER, SAN FRANCISCO, CA 94105, UNITED STATES","26 PIER, SAN FRANCISCO, CA, 94111 | 26 PIER, S...",37.795961269966 | 37.740561465372,-122.395085944366 | -122.376715590602,060750105002002 | 060759809001017
3,"256 EL CAMINO REAL, SAN MATEO, CA 94401, UNITE...","256 S EL CAMINO REAL, SAN MATEO, CA, 94401 | 2...",37.563454197511 | 37.568959419958,-122.326559076275 | -122.333189329068,060816064005002 | 060816059022004
4,"100 SKYLINE PLZ, DALY CITY, CA 94015, UNITED S...","100 SKYLINE BLVD, DALY CITY, CA, 94015 | 100 S...",37.673925650836 | 37.691764185401,-122.488596769759 | -122.494884037259,060816015011000 | 060816010004000
5,"PIER 5 PACIFIC AVE, SAN FRANCISCO, CA 94105, ...","5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 | 5 PA...",37.798097684223 | 37.796195392821,-122.397413747842 | -122.412913979,060750105002006 | 060750108002001
6,"435 BROADWAY BLVD, SAN FRANCISCO, CA 94133, UN...","435 BROADWAY ST, SAN FRANCISCO, CA, 94133 | 43...",37.798087228536 | 37.798087228536,-122.404443200708 | -122.404443200708,060750106002006 | 060750106002006
7,"88 N HILL DR, BRISBANE, CA 94005, UNITED STATES","88 S HILL DR, BRISBANE, CA, 94005 | 88 W HILL ...",37.686556712518 | 37.692860513616,-122.411619079175 | -122.421788578945,060816001002017 | 060816001002013
8,"313 SAN MATEO DR, SAN MATEO, CA 94401, UNITED ...","313 S SAN MATEO DR, SAN MATEO, CA, 94401 | 313...",37.564197193327 | 37.571051391926,-122.323970606746 | -122.331329802994,060816063003022 | 060816059021003
9,"751 SAN BRUNO AVE, SAN BRUNO, CA 94066, UNITED...","751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 | 75...",37.628528342383 | 37.630969151044,-122.417215794707 | -122.406114264877,060816040001002 | 060816042001004


In [5]:
import folium
from math import radians, sin, cos, sqrt, atan2

# Parse pipe-separated lat/lon values
def parse_pipe(val):
    return [v.strip() for v in str(val).split('|')]

df['lat1'] = df['tie_latitudes'].apply(lambda x: float(parse_pipe(x)[0]))
df['lat2'] = df['tie_latitudes'].apply(lambda x: float(parse_pipe(x)[1]))
df['lon1'] = df['tie_longitudes'].apply(lambda x: float(parse_pipe(x)[0]))
df['lon2'] = df['tie_longitudes'].apply(lambda x: float(parse_pipe(x)[1]))

# Haversine distance in km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

df['distance_km'] = df.apply(lambda r: haversine(r.lat1, r.lon1, r.lat2, r.lon2), axis=1)

# Map centered on all points
center_lat = (df['lat1'].mean() + df['lat2'].mean()) / 2
center_lon = (df['lon1'].mean() + df['lon2'].mean()) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

colors = ['red', 'blue', 'green', 'purple', 'orange']
for i, row in df.iterrows():
    color = colors[i % len(colors)]
    dist = row['distance_km']
    label = row['full_address_std']

    folium.CircleMarker(
        [row.lat1, row.lon1], radius=7, color=color, fill=True, fill_opacity=0.9,
        popup=folium.Popup(f"<b>{label}</b><br>Match 1<br>Distance: {dist:.3f} km", max_width=300)
    ).add_to(m)
    folium.CircleMarker(
        [row.lat2, row.lon2], radius=7, color=color, fill=False,
        popup=folium.Popup(f"<b>{label}</b><br>Match 2<br>Distance: {dist:.3f} km", max_width=300)
    ).add_to(m)
    folium.PolyLine(
        [[row.lat1, row.lon1], [row.lat2, row.lon2]],
        color=color, weight=2, tooltip=f"{label}: {dist:.3f} km"
    ).add_to(m)

display(df[['full_address_std', 'distance_km']].rename(columns={'full_address_std': 'Address', 'distance_km': 'Distance (km)'}))
m

,Address,Distance (km)
0,"1485 BAYSHORE BLVD, SAN FRANCISCO, CA 94124, U...",0.000000
1,"3130 LA SELVA DR, SAN MATEO, CA 94403, UNITED ...",0.071875
2,"26 PIER, SAN FRANCISCO, CA 94105, UNITED STATES",6.368293
3,"256 EL CAMINO REAL, SAN MATEO, CA 94401, UNITE...",0.846305
4,"100 SKYLINE PLZ, DALY CITY, CA 94015, UNITED S...",2.059274
5,"PIER 5 PACIFIC AVE, SAN FRANCISCO, CA 94105, ...",1.378250
6,"435 BROADWAY BLVD, SAN FRANCISCO, CA 94133, UN...",0.000000
7,"88 N HILL DR, BRISBANE, CA 94005, UNITED STATES",1.136691
8,"313 SAN MATEO DR, SAN MATEO, CA 94401, UNITED ...",1.000790
9,"751 SAN BRUNO AVE, SAN BRUNO, CA 94066, UNITED...",1.014612
